### Crie uma tabela de Produtos unindo Categoria (Categories), Marca (brands) e Quantidade Total (Order_Items) de Estoque (Stocks) em todas as Lojas (Stores)

In [0]:
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
# Criando várias tabelas temporárias 
bronze_map = {
    'tmp_bronze_brands': f'{bronze_path}/brands/',
    'tmp_bronze_categories': f'{bronze_path}/categories/',
    #'tmp_bronze_customers': f'{bronze_path}/customers/',
    #'tmp_bronze_order_items': f'{bronze_path}/order_items/',
    #'tmp_bronze_orders': f'{bronze_path}/orders/',
    'tmp_bronze_products': f'{bronze_path}/products/',
    #'tmp_bronze_staffs': f'{bronze_path}/staffs/',
    'tmp_bronze_stocks': f'{bronze_path}/stocks/',
    #'tmp_bronze_stores': f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

#### Será realizado a soma de estoque em todas as lojas CTE (Common Table Expression) é um "bloco temporário de consulta" dentro do SQL

In [0]:
df_product_silver = spark.sql("""
-- Realizando o CTE
WITH stock AS (
SELECT 
    product_id,
    sum(quantity) AS Total_Stock
FROM tmp_bronze_stocks 
GROUP BY product_id
)

-- Consulta principal
SELECT
    p.product_id,
    p.product_name,
    p.brand_id,
    -- b.brand_id AS brand_id_marca,
    -- b.brand_name,
    -- p.category_id AS category_id_produto,
    -- c.category_id AS category_id_categoria,
    c.category_name,
    p.model_year,
    p.list_price,
    s.total_stock
FROM tmp_bronze_products AS p
LEFT JOIN tmp_bronze_categories AS c ON p.category_id = c.category_id
LEFT JOIN     tmp_bronze_brands AS b ON p.brand_id = b.brand_id
LEFT JOIN                 stock AS s ON p.product_id = s.product_id                                                 
""")
# Salvar e transformar em formato delta na camada Silver
df_product_silver.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{silver_path}/product')


In [0]:
%sql
/* Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)
CREATE TABLE IF NOT EXISTS bikestore.logistics.silver_product
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/product';*/




In [0]:
%sql
-- SELECT * FROM bikestore.logistics.silver_product LIMIT 15